[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lmassaron/gemma_from_scratch/blob/main/workshop/06_rmsnorm_and_normalization.ipynb)


# Workshop: Building Gemma 3 from Scratch


[<- Previous: 05 Gated MLP](05_gated_mlp.ipynb) | [Next: 07 The Transformer Block ->](07_the_transformer_block.ipynb)

## Notebook 6: RMSNorm and Double Normalization

**Estimated Time: 10 minutes**

Modern Gemma models use **Root Mean Square Layer Normalization (RMSNorm)**. It is simpler and faster than standard LayerNorm because it skips the mean calculation — only computing RMS (root mean square).

---

## Learning Objectives:
1. Understand the math of RMSNorm.
2. Implement the Gemma-specific "Add-One" RMSNorm layer.
3. Learn about the **"double-norm"** (Pre + Post) for every sub-layer — Gemma's signature trick.

In [1]:
import torch
import torch.nn as nn
import sys

print(f"Python version: {sys.version}")
print(f"PyTorch version: {torch.__version__}")

dim = 768  # Gemma 3: hidden_size
eps = 1e-6

Python version: 3.12.0 (main, Oct  2 2023, 20:56:14) [Clang 16.0.3 ]
PyTorch version: 2.12.0


---

## 1. RMSNorm Math

---

## LayerNorm vs RMSNorm

**LayerNorm**: $y = \frac{x - E[x]}{\sqrt{Var[x] + \epsilon}} \cdot \gamma + \beta$

**RMSNorm**: $y = \frac{x}{\sqrt{Mean(x^2) + \epsilon}} \cdot \gamma$

Research shows that the mean-centering part of LayerNorm is not strictly necessary for model stability. By computing only the Root Mean Square (RMS), Gemma 3 saves compute cycles while still keeping the vector's activation scale within a healthy, unit-variance range.

RMSNorm skips the mean subtraction ($E[x]$), which is a simplification that works well in practice. The weight $\gamma$ is a learned parameter vector that scales the normalized output.

---

## ⚖️ RMSNorm Mathematics & the Gemma "Add-One" Initialization

Commercial language models traditionally used **LayerNorm** to stabilize activations. Standard LayerNorm (which subtracts the mean) is computationally expensive, so Gemma 3 uses the more efficient **RMSNorm (Root Mean Square Normalization)** instead, which is simpler and faster.

### 🧮 LayerNorm vs RMSNorm Formulas:
- **LayerNorm** computes the mean and variance to normalize activations:

  $$y = \frac{x - \mu}{\sqrt{\sigma^2 + \epsilon}} \odot \gamma + \beta$$

- **RMSNorm** drops the mean subtraction ($\mu=0$) and bias ($\beta=0$), scaling only by the Root Mean Square:

  $$\text{RMSNorm}(x) = \frac{x}{\text{RMS}(x)} \odot (1 + \gamma) = \frac{x}{\sqrt{\frac{1}{d} \sum_{i=1}^d x_i^2 + \epsilon}} \odot (1 + \gamma)$$

### 🛠️ The Gemma "Add-One" Trick:
Standard RMSNorm initializes scale weights $\gamma$ to $1.0$. However, Gemma 3 initializes $\gamma$ to **zero** and uses the scale factor **$(1 + \gamma)$**. 
- At step zero, $\gamma = 0$, so $(1 + 0) = 1.0$. This means the block acts as an **exact identity pass-through** at initialization.
- This prevents early training instability (at the start of training, the model doesn't suffer from random weight scaling) by allowing signals and gradients to flow freely before parameters diverge from zero, ensuring smoother convergence!

In [2]:
class RMSNorm(nn.Module):
    """Gemma 3-style RMSNorm with Add-One initialization."""

    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        # Gemma's Add-One trick: weight starts at zero
        self.weight = nn.Parameter(torch.zeros(dim))

    def _norm(self, x):
        # x: (... , dim)
        # RMS = sqrt(mean(x^2))
        # return x / RMS
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)

    def forward(self, x):
        # Apply (1 + weight) scaling — at init this is just 1.0 (identity)
        return self._norm(x.float()).type_as(x) * (1.0 + self.weight)


norm = RMSNorm(dim)
x = torch.randn(2, 5, dim) * 10  # High variance input
out = norm(x)

print(f"Input variance:  {x.var().item():.4f}")
print(f"Output variance: {out.var().item():.4f}")

# At initialization, weight=0 so output is just normalized (variance ~1)
print(f"Weight: {norm.weight.data[:3].tolist()}... (all zeros)")
print(f"Effective scale: {(1 + norm.weight.data[0]).item():.4f}")
assert out.shape == x.shape
print("✅ RMSNorm works correctly!")

Input variance:  98.3657
Output variance: 1.0001
Weight: [0.0, 0.0, 0.0]... (all zeros)
Effective scale: 1.0000
✅ RMSNorm works correctly!


---

## 2. The "Double-Norm" Secret: Pre-Norm + Post-Norm on EVERY sub-layer

There is a structural choice in Gemma 3 which is unique compared to Llama or Mistral: **every** sub-layer gets **both** pre-norm and post-norm. This is "double normalization".

Most modern models use Pre-Norm (normalizing the input to the sub-layer). Gemma 3, however, places an RMSNorm layer both before and after every attention and MLP sub-layer.

This approach makes the model incredibly resistant to training instability. Even if the Attention layer produces a noisy or slightly skewed result, the Post-Norm "tames" it before it can corrupt the signal in the residual stream, allowing for deeper networks.

```python
# For the Attention sub-layer:
x = input 
x = RMSNorm_pre(x)           # normalize before
x = Attention(x)             # apply
x = x + RMSNorm_post(x)     # normalize + residual  <-- unique!

# For the MLP sub-layer:
x = RMSNorm_pre(x)           # normalize before
x = MLP(x)                   # apply
x = x + RMSNorm_post(x)     # normalize + residual  <-- unique!
```

---

## 3. Why Double-Norm Works So Well

Comparing architectures:

| Architecture | Norm locations | Stability |
|---|---|---|
| **Post-norm** (original Transformer) | After each sub-layer only | Unstable at depth |
| **Pre-norm** (T5, Llama) | Before each sub-layer only | Moderate |
| **Double-norm** (Gemma 3/3) | Pre + Post on **every** sub-layer | **Most stable** |

The post-norm is the innovation that sets Gemma apart from Llama.

---

## Exercise:
Why does initializing the weight to zero and using $(1 + weight)$ help with training stability compared to initializing the weight to one and using it directly?

In [3]:
# Answer: The model starts in identity mode
init_weight = torch.zeros(dim)
init_scale = 1.0 + init_weight[0]  # = 1.0
print(f"At initialization: scale = {init_scale:.1f} (exactly 1.0)")

# The layer is essentially: RMSNormalize(x) * 1.0 = RMSNormalize(x)
# The signal flows through untouched — gradients are not skewed by random scaling initially.
print("\nThe layer acts as RMS-normalization-only at the start.")
print("Gradients can propagate freely before norm parameters diverge from zero.")

At initialization: scale = 1.0 (exactly 1.0)

The layer acts as RMS-normalization-only at the start.
Gradients can propagate freely before norm parameters diverge from zero.


---

## Key Takeaway

Gemma 3's signature normalization: **Double RMSNorm** (Pre + Post on every sub-layer) with **Add-One** weight initialization. This makes Gemma 3 incredibly stable even with deep architectures.

